In [ ]:
### From segmentation framework ###

## OK this is kind of take 3.
## segmentation is hard for dataset BBBC06 because some nuclei are elongated and many are touching

## randomly trying different steps wasn't working and is not really the right approach

## we should have a framework or decision tree type approach that says
## Step 1. if you have xx, then try yy.
## Step 2. If it looks like this, then do that.

## So here is an attempt at attempting to use that framework
## https://chatgpt.com/c/6909e48e-96bc-832b-8c9f-8a4653a0c959

## Libraries
## Load packages
# import glob, os
import skimage
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pandas as pd
import os
# import napari_segment_blobs_and_things_with_membranes as nsbatwm
# import napari_simpleitk_image_processing as nsitk
# import scipy

## set dirs
cwd = Path.cwd()
# Find the first parent that matches the target folder
for p in cwd.parents:
    if p.name == "UCL-Biosciences-Image-Analysis":
        DIR = p
        break
else:
    raise FileNotFoundError("Base directory 'UCL-Biosciences-Image-Analysis' not found in path.")

DATA_DIR = DIR / "input_data" / "BBBC006_v1_images_z_16"


# 2D Segmentation Decisions
Segmentation of 2D bioimages is a decision process, not a single algorithm. Classic (non-deep-learning) methods work well for many fluorescence datasets, but performance depends on contrast, object size, touching rate, and staining consistency. This guide provides a structured workflow and routing logic so users do not jump randomly between filters and thresholds.

The framework follows a simple sequence:

1. Define object and constraints
2. Preprocess (illumination, noise, contrast)
3. Extract foreground
4. Generate seeds (if instances touch)
5. Separate objects (watershed)
6. Filter and correct
7. Quality-check and iterate

Each stage has clear decision points. If the required conditions are not met, the user branches appropriately (e.g. improve contrast or adjust seeding rather than forcing a failing threshold). The aim is to reduce guesswork and keep workflows simple and reproducible.

This is not a code cookbook. It is a method to choose the right steps and avoid unnecessary complexity. If the image fails early checks and cannot be rescued with standard preprocessing, acquisition improvements should be prioritised rather than stacking more operations.

### 1. Define object and constraints
this is for the user to understand the data and what the data mean for the analysis

## What to record up front ##

Object: nuclei / whole cells / sub-objects

Imaging: fluorescence / brightfield; channels used for object vs boundary

Size range: expected pixel area or diameter. Calculate:

- effective_px_size = camera_px_size / magnification
- effective_px_size_after_binning = effective_px_size × bin_factor
- expected_pixel_diameter = physical_diameter / effective_px_size_after_binning

Touching frequency: low / medium / high

Shape: round / elongated / irregular

Signal reliability: consistent / variable (across field, across batch)

Error tolerance: more acceptable to miss objects or to over-split?

## Why this matters ##
Size → filters and distance-transform parameters

Touching → need for seeds and watershed

Shape → distance peaks vs erosion vs skeleton seeds

Signal quality → global vs adaptive thresholding

Error preference → threshold and watershed aggressiveness

# If missing information
If you cannot estimate size and touching rate, inspect ~50 fields and measure.

If you do not know whether contrast is stable, check across plates/batches.

## Stop rule ##
If contrast is fundamentally insufficient after basic preprocessing, do not proceed. Fix staining/illumination.


In [ ]:
### Read in data
# List files
files = sorted(DATA_DIR.glob("*_w1*.tif"))[ :100]
files

### for a list of tifs
images = [ ]
for file in files:
    images.append(skimage.io.imread(file))
# skimage.io.imread_collection(files)

print(images[0].shape)
print('we have ', len(images), ' images')
print('each image has ', images[0].shape, ' pixels')

fig,ax = plt.subplots(1, 4, figsize = (30, 60) )
for i in range(0,4):
    ax[i].imshow(images[i])

## when setting up the analysis, we will use one image at a time, defined here:
img = images[3]

## Fill in for your dataset

We are working with this dataset: https://bbbc.broadinstitute.org/BBBC006. Using specifically plane 16, which is in focus, and w1 (Hoechst 33342 stain for DNA)

**Object**: Nuclei (primary). Whole-cell possible via phalloidin.

**Imaging**: Fluorescence, DAPI/Hoechst (w1) + phalloidin (w2)

**Size range**: 
- U2OS cell nuclei ~16um width and 23um length. (https://pmc.ncbi.nlm.nih.gov/articles/PMC4227890/).
- Pixel size of the camera is 6.45um and magnification is 20x, meaning each pixel is 6.45um/20 = 0.3225.  The data are binned 2x, which means each (super) pixel is 0.645um
- So we expect nuclei to be 16/.645 x 23/.645 ~ 25 x 36 pixels.

**Touching frequency**: medium. Looking at raw images shows many, but not all, objects are touching.

**Shape**: Many round, some elongated

**Signal reliability**: Using in-focus, signal is good.

**Primary failure to avoid**: oversegmenting elongated nuclei, undersegmenting touching nuclei.



In [ ]:
### 2. Pre-processing 
# i. Correct uneven background.
# contrast between objects and background is easier to identify when background is even

# ii. Denoise. 
# use median or gaussian filter to reduce noise and make contrasts more stable

# iii. Normalise intensity
# Brightness varies among images. Normalise so steps are applicable across images.

# iv. Visualise
# does it look sensible?


### i. Correct background
# use rolling ball algorithm from skimage
from skimage import restoration, morphology

### i. Correct background
# use rolling ball algorithm from skimage
from skimage import restoration, morphology

# write bg correction function
def rolling_ball_background_correction(img, radius):
    # estimate background
    background = morphology.opening(img, morphology.disk(radius)) # radius > size of objects to be detected
    # subtract background from original image
    img_bg_corrected = img - background
    # ensure no negative values
    img_bg_corrected = np.clip(img_bg_corrected, 0, None)
    bg_corrected = morphology.opening(img_bg_corrected, morphology.disk(radius))
    return img_bg_corrected, background, bg_corrected

img_bg_corrected, background, bg_corrected = rolling_ball_background_correction(img, radius=40)

# define visualise function for multiple panels
def visualise_images(images, titles):
    n = len(images)
    fig, ax = plt.subplots(1, n, figsize=(5 * n, 5))
    for i in range(n):
        ax[i].imshow(images[i], cmap='gray')
        ax[i].set_title(titles[i])
    plt.show()

visualise_images(
    [img, background, img_bg_corrected, bg_corrected],
    ['Original Image', 'Estimated Background', 'Background Corrected Image', 'Background From Corrected Image']
)

# Background should be smooth (panel 2) and not show cell-like structure
# Background corrected image (panel 3) should have improved contrast and nuclei should not be hollow
# Background from corrected image (panel 4) should be consistant and ideally consistently dark 

In [ ]:
# Pre-processing ii. Denoise
# Apply Gaussian filter
from scipy.ndimage import gaussian_filter

# denoise function
def denoise_image(img, method='gaussian', **kwargs):
    if method == 'gaussian':
        sigma = kwargs.get('sigma', 1)
        return gaussian_filter(img, sigma=sigma)
    elif method == 'median':
        size = kwargs.get('size', 3)
        return skimage.filters.median(img, selem=skimage.morphology.disk(size))
    else:
        raise ValueError("Unsupported denoising method")

# Apply denoising
img_denoised = denoise_image(img_bg_corrected, method='gaussian', sigma=1)

# visualise denoising
visualise_images(
    [img_bg_corrected, img_denoised],
    ['Before Denoising', 'After Denoising']
)

# Denoised image (panel 2) should have reduced noise while preserving nuclei structure


In [ ]:
# Pre-processing iii. Normalise intensity
# normalisation needed if exposure varies among images

# here we compare at median, 5th and 95th percentiles of each image
raw_intensity_stats = []
for image in images:
    p5 = np.percentile(image, 5)
    p95 = np.percentile(image, 95)
    median = np.median(image)
    raw_intensity_stats.append((p5, median, p95))    
# Convert to numpy array for easier plotting
raw_intensity_stats = np.array(raw_intensity_stats)

# Plot raw intensity stats
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(raw_intensity_stats[:, 0], 'o-', label='5th Percentile')
ax.plot(raw_intensity_stats[:, 1], 'o-', label='Median')
ax.plot(raw_intensity_stats[:, 2], 'o-', label='95th Percentile')
ax.set_xlabel('Image Index')
ax.set_ylabel('Intensity Value')
ax.set_title('Raw Intensity Statistics Across Images')
ax.legend()
plt.show()

# in this case, the median brightness is similar across images
# so we don't normalise


In [ ]:
### Extract Foreground
# create a binary mask, with objects as 1 and background as 0

# here you could use a global threshold which is applied across the whole image
# or an adaptive threshold which calculates a local threshold for each pixel based on its neighbourhood

# because we have reduced background noise, we can try a global threshold first

from skimage.filters import threshold_otsu
# Calculate global threshold using Otsu's method

# define function for generating binary mask
def generate_binary_mask(img, method='global', **kwargs):
    if method == 'global':
        thresh = threshold_otsu(img)
        binary_mask = img > thresh
    elif method == 'adaptive':
        block_size = kwargs.get('block_size', 35)
        offset = kwargs.get('offset', 10)
        local_thresh = skimage.filters.threshold_local(img, block_size, offset=offset)
        binary_mask = img > local_thresh
    else:
        raise ValueError("Unsupported thresholding method")
    return binary_mask

# Generate binary mask using global threshold
binary_mask = generate_binary_mask(img_denoised, method='global')

# visualise binary mask
visualise_images(
    [img_denoised, binary_mask],
    ['Denoised Image', 'Binary Mask (Global Threshold)']
)

# Binary mask (panel 2) should show clear separation of nuclei from background
# has it caught even the dimmest nuclei?

## 4. Generate Seeds
Seeding forces the algorithm to start one region per object. Without seeds, touching nuclei merge and thresholding gives blobs, not individual objects. Seeds tell watershed “split here”.

A seed is a single marker pixel or small region placed inside each object. Watershed then grows each seed outward until object boundaries meet.

### Main seeding options (non-DL)

1) *Distance-transform peaks (default)*

Compute distance inside mask → peaks ≈ nucleus centres. Works when nuclei roughly round. Tunable through min distance, and h-max. We generally use this first.

2) *Morphological erosion seeds*

Erode mask to break touches, label components, then dilate seeds back. Robust in dense fields. Risks missing small/faint nuclei if erosion too strong.

3) *LoG (Laplacian-of-Gaussian) blob seeds*

Detect bright blobs directly on intensity image. Good when mask imperfect or nuclei elongated. Needs σ chosen to match object size.

4) *Manual or semi-manual* (rare)

For quality control or difficult edge cases only.

### How to choose
Round nuclei, good mask-        	Distance peaks

Crowded nuclei that stay fused -	Erode mask then seed

Mask imperfect, variable shapes -  	LoG blobs

Very faint nuclei -             	Lower threshold + DT, or small erosion

### Quality targets

- 1 seed per nucleus
- Few extras (filtered by later size rules)
- Rare doubles ok (watershed can merge small ones)
- No seeds in background

Keep it simple: start with distance-transform peaks. Only switch if failure mode is clear and consistent.

In [ ]:
# our mask is pretty good, not to crowded and only some are touching
# so we will start with distance-based seeds

### Generate seeds with distance transform
from scipy import ndimage as ndi
from skimage import morphology, feature, measure

# define function for generating seeds
def generate_distance_transform_seeds(binary_mask, h=2, footprint_size=5, min_distance=5):
    # Distance transform inside mask
    distance = ndi.distance_transform_edt(binary_mask)

    # suppress tiny spurious peaks
    distance_h = morphology.h_maxima(distance, h=h)

    # peak_local_max now returns coordinates only
    coords = feature.peak_local_max(
        distance_h,
        footprint=morphology.disk(footprint_size),
        exclude_border=True,
        min_distance=min_distance
    )

    # Create empty seed mask
    seeds = np.zeros_like(binary_mask, dtype=np.int32)

    # Mark seeds at those coordinates
    for i, (r, c) in enumerate(coords, start=1):
        seeds[r, c] = i

    seeds = measure.label(seeds)
    return seeds

# Generate seeds
seeds = generate_distance_transform_seeds(binary_mask, h=2, footprint_size=5)

# Visualise seeds 

# define visualise function for seeds
def visualise_seeds_on_mask(binary_mask, seeds):
    plt.figure(figsize=(8, 8))
    plt.imshow(binary_mask, cmap='gray')
    ys, xs = np.where(seeds > 0)
    plt.scatter(xs, ys, s=10, c='red')
    plt.title("Seeds on Mask")
    plt.axis('off')
    plt.show()

visualise_seeds_on_mask(binary_mask, seeds)

In [ ]:
### not great. lots of objects have multiple seeds

# increasing h in h_maxima to suppress more peaks
# increasing disc in footprint to require larger separation between peaks

# Distance transform inside mask
seeds_dist4_fp7 = generate_distance_transform_seeds(binary_mask, h=4, footprint_size=7)

# Visualise seeds 
visualise_seeds_on_mask(binary_mask, seeds_dist4_fp7)

In [ ]:
# still no good.
# we will try setting the min_distance parameter in peak_local_max to 10

seeds_dist4_fp7_md10 = generate_distance_transform_seeds(binary_mask, h=4, footprint_size=7, min_distance=10)

# Visualise seeds
visualise_seeds_on_mask(binary_mask, seeds_dist4_fp7_md10)

# by setting min_distance we have got a better distribution of seeds
# it is not perfect
# one nucleus has two seeds
# a few touching nuclei still have only one seed
# but overall it is much improved

# we will proceed with these seeds for watershed segmentation
# and try to improve segmentation with post-processing

In [ ]:
# 5. Separate objects using watershed segmentation
# Perform watershed segmentation to separate touching objects in a binary mask 
# using distance transform and predefined seeds.

from skimage.segmentation import watershed

# Apply watershed segmentation
labels = watershed(img, seeds, mask=binary_mask)

# Visualise segmentation showing boundaries
from skimage.segmentation import find_boundaries

bound = find_boundaries(labels, mode='outer')

plt.figure(figsize=(5,5))
plt.imshow(img, cmap='gray')          # original or background-corrected
plt.contour(labels, colors='yellow', linewidths=0.5)
plt.title("Watershed contours on nuclei")
plt.axis('off')

# again this looks sensible
# but we will need to correct errors



In [ ]:
# 6. Post-process segmentation to correct errors
# we want to correct nuclei that are:
# - oversegmented (split into multiple parts)
# - undersegmented (merged together)
# - remove nuclei touching the border (these are not complete)

from skimage import measure, filters, segmentation

# remove nuclei touching the border
labels_cleared = segmentation.clear_border(labels)

# remove small objects, including over segmentation fragments
from skimage.morphology import remove_small_objects
labels_no_small = remove_small_objects(labels_cleared, min_size=150)

# Visualise final segmentation
plt.figure(figsize=(5,5))
plt.imshow(img, cmap='gray')          # original or background-corrected
plt.contour(labels_no_small, colors='red', linewidths=0.5)
plt.title("Final Segmentation after Post-processing")
plt.axis('off')
plt.show()

# we don't have any nuclei touching the border or tiny fragments
# final problem is the undersegmented nuclei

In [ ]:
### Undersegmented nuclei
# we can try to identify these based on shape and size
# get properties: label, size, shape (e.g., eccentricity, solidity)
properties = measure.regionprops_table(labels_no_small,
                                       properties=['label',
                                                   'area',
                                                   'eccentricity',
                                                   'solidity'])
properties = pd.DataFrame(properties)

# define thresholds for undersegmentation based on size and shape distributions
# plot histograms
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].hist(properties['area'], bins=30, color='blue', alpha=0.7)
ax[0].set_title('Nuclei Area Distribution')
ax[0].set_xlabel('Area (pixels)')
ax[0].set_ylabel('Frequency')
ax[1].hist(properties['eccentricity'], bins=30, color='green', alpha=0.7)
ax[1].set_title('Nuclei Eccentricity Distribution')
ax[1].set_xlabel('Eccentricity')
ax[1].set_ylabel('Frequency')
ax[2].hist(properties['solidity'], bins=30, color='orange', alpha=0.7)
ax[2].set_title('Nuclei Solidity Distribution')
ax[2].set_xlabel('Solidity')
ax[2].set_ylabel('Frequency')
plt.show()


In [ ]:
# based on the distributions, the undersegmented nuclei will be
# - larger than average size
# - less elongated (higher eccentricity)
# - less solid (lower solidity)
size_thresh = properties['area'].mean() + ( 2 * properties['area'].std() )
eccentricity_thresh = 0.5
solidity_thresh = properties['solidity'].mean() - (2 * properties['solidity'].std() )

# list to hold labels of undersegmented nuclei
undersegmented_labels = []

# identify undersegmented nuclei
for _, row in properties.iterrows():
    if (row['area'] > size_thresh or
        # row['eccentricity'] < eccentricity_thresh or
        row['solidity'] < solidity_thresh):
        undersegmented_labels.append(row['label'])

# Visualise undersegmented nuclei
plt.figure(figsize=(5,5))
plt.imshow(img, cmap='gray')          # original or background-corrected
plt.contour(labels_no_small, colors='red', linewidths=0.5)
for label in undersegmented_labels:
    coords = np.array(np.where(labels_no_small == label)).T
    plt.scatter(coords[:, 1], coords[:, 0], s=1, c='blue')
plt.title("Undersegmented Nuclei Highlighted")
plt.axis('off')
plt.show()


In [ ]:
### we might be able to improve this by eroding the mask before seeding and using watershed
# Erode mask to separate touching nuclei
from skimage.morphology import binary_erosion, disk
# disk argument defines the size of the erosion
# should be tuned based on expected nucleus size and degree of touching
# for nuclei 25-35 pixels in diameter, we will try 2, 4, 6 pixels
eroded_mask_2 = binary_erosion(binary_mask, disk(2))
eroded_mask_4 = binary_erosion(binary_mask, disk(4))
eroded_mark_6 = binary_erosion(binary_mask, disk(6))

# visualise eroded masks
plt.subplots(1, 3, figsize=(20, 5))
plt.subplot(1, 3, 1)
plt.imshow(eroded_mask_2, cmap='gray')
plt.title('Eroded Mask (2 px)')
plt.axis('off')
plt.subplot(1, 3, 2)
plt.imshow(eroded_mask_4, cmap='gray')
plt.title('Eroded Mask (4 px)')
plt.axis('off')
plt.subplot(1, 3, 3)
plt.imshow(eroded_mark_6, cmap='gray')
plt.title('Eroded Mask (6 px)')
plt.axis('off')
plt.show()

# interesting. erosion is making the nuclei smaller but not removing the bridges between them
# we will try seeding and watershed on the eroded masks anyway

In [ ]:
## Generate seeds on eroded mask (2 px)
# Distance transform inside eroded mask
distance_eroded = ndi.distance_transform_edt(eroded_mask_2)

# suppress tiny spurious peaks
distance_h_eroded = morphology.h_maxima(distance_eroded, h=3)
# peak_local_max now returns coordinates only
coords_eroded = feature.peak_local_max(
    distance_h_eroded,
    footprint=morphology.disk(5),  # adjust 3–7
    min_distance=10 # < half expected diameter
)
# Create empty seed mask
seeds_eroded = np.zeros_like(binary_mask, dtype=np.int32)
# Mark seeds at those coordinates
for i, (r, c) in enumerate(coords_eroded, start=1):
    seeds_eroded[r, c] = i
seeds_eroded = measure.label(seeds_eroded)

# watershed on eroded mask
labels_eroded = watershed(-distance_eroded, seeds_eroded, mask=eroded_mask_2)

# watershed from eroded seeds but original mask
labels_eroded_full_mask = watershed(-distance_eroded, seeds_eroded, mask=binary_mask)


# visualise seeds and watershed on eroded mask in subplots
plt.subplots(1, 3, figsize=(20, 5))
plt.subplot(1, 3, 1)
plt.imshow(eroded_mask_2, cmap='gray')
ys, xs = np.where(seeds_eroded > 0)
plt.scatter(xs, ys, s=10, c='red')
plt.title("Seeds on Eroded Mask (2 px)")
plt.axis('off')
plt.subplot(1, 3, 2)
plt.imshow(eroded_mask_2, cmap='gray')
plt.contour(labels_eroded, colors='red', linewidths=0.5)
plt.title("Watershed on Eroded Mask (2 px)")
plt.axis('off')
plt.subplot(1, 3, 3)
plt.imshow(img, cmap='gray')          # original or background-corrected
plt.contour(labels_eroded_full_mask, colors='red', linewidths=0.5)
plt.title("Final Segmentation on Eroded Mask on original image (2 px)")
plt.axis('off')
plt.show()

# hasn't improved :'D

## Conclusion
In the end, segmenting touching objects is difficult. These days, the best option may well be to use cellpose or stardist to segment them, as these algorithms tend to perform better than trying different python-based parameters.


## Quantification
Worth comparing against ground truth anyway. Let's count cells for each image.

From the BBBC description: "For each of the 768 fields of view (384 wells, 2 fields of view per well), an automated algorithm used the image at optimal focus (z = 16) to identify and count the nuclei"

In [ ]:
### count nuclei in final segmentation
#

final_labels = skimage.segmentation.clear_border(labels_no_small)
num_nuclei = len(np.unique(final_labels)) - 1  # subtract 1 for background
print("Number of nuclei detected:", num_nuclei)



In [ ]:
## Now we have a full framework for segmentation in one image
## now we will make a function that we can apply to all images in a loop
## using all the steps above
def segment_nuclei(img):
    # Pre-processing
    img_bg_corrected, _, _ = rolling_ball_background_correction(img, radius=40)
    img_denoised = denoise_image(img_bg_corrected, method='gaussian', sigma=1)

    # Generate binary mask
    binary_mask = generate_binary_mask(img_denoised, method='global')

    # Generate seeds
    seeds = generate_distance_transform_seeds(binary_mask, h=4, footprint_size=7, min_distance=10)

    # Watershed segmentation
    labels = watershed(img, seeds, mask=binary_mask)

    # Post-process segmentation
    labels_cleared = segmentation.clear_border(labels)
    labels_no_small = remove_small_objects(labels_cleared, min_size=150)

    # count number of nuclei
    num_nuclei = len(np.unique(labels_no_small)) - 1  # subtract 1 for background

    return num_nuclei

In [ ]:
nuc_counts = []

## we don't load all images - see cell at top. Currently just 20 out of several hundred!
for i, img in enumerate(images):
    print(i)
    nuc_counts.append({'file' : files[i].name, 'nuc_count' : segment_nuclei(img)})

nuc_counts = pd.DataFrame(nuc_counts)

In [ ]:
## load the ground truth data
gt_counts_file = DATA_DIR / "BBBC006_results_bray.csv"

gt_counts_df = pd.read_csv(gt_counts_file)
# file names are in gt_counts_df['Image_FileName_OrigDAPI']
gt_counts_df.rename(columns={'Image_FileName_OrigDAPI': 'file'}, inplace=True)

In [ ]:
### merge ground truth and estimated nuclei counnts
merged_counts = pd.merge(nuc_counts, gt_counts_df[['file', 'Image_Count_Nuclei']], how='left')
merged_counts.rename(columns={'nuc_count': 'skimage_count',
                              'Image_Count_Nuclei' : 'ground_truth_count'}, inplace=True)
## save the table to new output dir in repo root
output_dir = DIR / "output"
os.makedirs(output_dir, exist_ok=True)
output_file = output_dir / "BBBC006_nuclei_counts_comparison.csv"
merged_counts.to_csv(output_file, index=False)

# plot nuclei count vs ground truth
plt.figure(figsize=(8, 6))
plt.scatter(merged_counts['ground_truth_count'], merged_counts['skimage_count'], alpha=0.7)
plt.plot([0, merged_counts['ground_truth_count'].max()], [0, merged_counts['ground_truth_count'].max()], 'r--')
plt.xlabel('Ground Truth Nuclei Count')
plt.ylabel('Estimated Nuclei Count')
plt.title('Estimated vs Ground Truth Nuclei Count')
plt.show()


In [ ]:
### Quantifying accuracy
# absolute value of the difference between estimated and ground truth counts
# divided by ground truth count to get percentage error
# then divide by number of images to get average error (and SD)
merged_counts['abs_error'] = np.abs(merged_counts['skimage_count'] - merged_counts['ground_truth_count'])
merged_counts['pct_error'] = merged_counts['abs_error'] / merged_counts['ground_truth_count'] * 100
mean_error = merged_counts['pct_error'].mean()
std_error = merged_counts['pct_error'].std()
print(f"Average Percentage Error: {mean_error:.2f}% ± {std_error:.2f}%")


# Next steps
It's not a terrible performance but the average error is too high to be acceptable. Interesting that the counts are consistently underestimated. Probably reflects difficulty separating touching nuclei. Next, we will try using a ML-trained model to see if it performs any better.